# Fractional Knapsack Problem (Greedy Approach)

## Problem Statement
We are given **n items**, each with a **value** and a **weight**, and a knapsack with a fixed **capacity**. The goal is to choose items (or *fractions* of items) so that the total value in the knapsack is **maximised** without exceeding the capacity.

## Fractional vs. 0/1 Knapsack
| Feature | Fractional Knapsack | 0/1 Knapsack |
|---|---|---|
| Can an item be split? | Yes | No |
| Best technique | Greedy | Dynamic Programming |
| Optimal with greedy? | Yes | No |

## Greedy Strategy
The greedy choice is to always pick the item with the **highest value-to-weight ratio** (value per unit weight) first:

1. Compute `value / weight` for every item.
2. Sort the items in **descending** order of this ratio.
3. Take whole items while they fit completely.
4. When the next item no longer fits, take only the **fraction** that fills the remaining capacity.

Because items can be split, this local best choice also gives the globally optimal answer.

## Step 1: Import Libraries
We use **NumPy** to store the values and weights as arrays and to combine them into a single 2-D structure (`np.column_stack`) so each item's value and weight stay paired while sorting.

In [ ]:
import numpy as np

## Step 2: The `greedyKnapsack` Function

**Signature:** `greedyKnapsack(value, weight, capacity)`

| Parameter | Meaning |
|---|---|
| `value` | NumPy array of item values |
| `weight` | NumPy array of item weights |
| `capacity` | Maximum weight the knapsack can hold |

**Returns:** the maximum total value (`pr`, short for *profit*).

### How it works, line by line
1. **Pair the data:** `np.column_stack((value, weight))` creates rows of `[value, weight]`, and `.tolist()` converts it to a Python list so it can be sorted with a custom key.
2. **Sort by ratio:** `temp.sort(key=lambda x: x[0]/x[1], reverse=True)` orders items from the best to the worst value-per-weight.
3. **Take whole items:** The loop walks through the sorted items. If an item's weight is **less than** the remaining capacity, it is taken fully:
   - `capacity -= weight`
   - `pr += value`
4. **Stop at the first item that doesn't fit:** `break` exits the loop, and `i` still points at that item.
5. **Take a fraction:** If capacity is still left, the fraction `capacity / weight` of item `i` is added: `pr += value * (capacity / weight)`.
6. **Return** the total profit.

### Notes
- The `print` statements show the **amount taken** and the **value of the item**. For example, `15.0 of 24.0` means "15 units of weight from the item whose value is 24".
- **Edge case:** if *every* item fits completely, the loop never breaks, and the final `if capacity > 0` block would add a fraction of the last item a second time. Guarding that block with a check such as `if i < len(value) and not all_taken` avoids this.

In [ ]:
def greedyKnapsack(value,weight,capacity):
            temp = np.column_stack((value,weight))
            temp = temp.tolist()
            temp.sort(key = lambda x: (x[0]/x[1]),reverse = True)
            pr =0
            i=0
            for i in range(len(value)):
                    if(capacity > 0 and temp[i][1]<capacity):
                            print(f"{temp[i][1]} of {temp[i][0]}")
                            capacity -= temp[i][1]
                            pr += temp[i][0]
                    else: break
            if(capacity>0):
                     pr += temp[i][0] * (capacity/temp[i][1])
                     print(f"{capacity/temp[i][1]} of {temp[i][0]}")
            return pr

## Step 3: Test the Function

### Input
| Item | Value | Weight | Ratio (V/W) |
|---|---|---|---|
| 1 | 25 | 18 | 1.39 |
| 2 | 24 | 15 | 1.60 |
| 3 | 15 | 10 | 1.50 |

**Knapsack capacity = 20**

### Expected walkthrough
1. Sorted by ratio: **Item 2 (1.60)**, **Item 3 (1.50)**, **Item 1 (1.39)**.
2. Item 2 (weight 15) fits fully, so the remaining capacity is `20 - 15 = 5` and the profit is `24`.
3. Item 3 (weight 10) does not fit in 5, so we take a fraction: `5 / 10 = 0.5` of it, adding `15 × 0.5 = 7.5`.
4. Total value = `24 + 7.5 = 31.5`.

In [ ]:

a = np.array([25,24,15],dtype = float)
b = np.array([18,15,10],dtype = float)

print(f"Maximised value: {greedyKnapsack(a,b,20.0)}")

## Results

```
15.0 of 24.0
0.5 of 15.0
Maximised value: 31.5
```

### Interpretation
- **Item with value 24:** taken **completely** (all 15 units of weight).
- **Item with value 15:** taken **50%** (5 of its 10 units of weight).
- **Item with value 25:** not taken at all, since it has the lowest ratio.
- **Maximum value achievable = 31.5**

This matches the hand calculation above, and because the greedy strategy is provably optimal for the fractional version, no other combination can beat 31.5.

## Complexity Analysis

Let **n** be the number of items.

| Step | Time |
|---|---|
| Building the `[value, weight]` list | O(n) |
| Sorting by ratio | **O(n log n)** |
| Single pass to fill the knapsack | O(n) |
| **Total** | **O(n log n)** |

**Space complexity: O(n)**, for the combined list `temp`. The sort uses O(n) auxiliary space in the worst case, and all other variables take O(1).

> The sorting step is the bottleneck. If the items are already sorted by ratio, the algorithm runs in O(n).